# Notebook 09: Full-catalog paired ground-motion fields

This notebook applies the validated Notebook 08 spatial factors to all
10,630 occurrences in the frozen 2,000,000-year annual catalog.

The output is a paired event-site table with three dependence cases:

- **I0_PHASE1_INDEPENDENT**: exact Phase 1 baseline;
- **C1_ALDEA22_SUBDUCTION**: primary correlated case;
- **C2_GODA_ATKINSON09**: correlation-model sensitivity.

Every case uses the same catalog occurrence, GMM median, tau, phi,
between-event residual, and two raw site-level latent vectors. Only the
within-event spatial transform changes. The 16 accepted Phase 1
partitions are reused for memory control and restartability.

In [ ]:
from __future__ import annotations

from collections import defaultdict
import gzip
import hashlib
import json
import math
import os
from pathlib import Path
import shutil
import sys
from typing import Any, Iterable

import numpy as np
import pandas as pd
from IPython.display import display


PIPELINE_VERSION = "notebook9_full_catalog_paired_ground_motion_v1"
EXPECTED_SITES = 470
EXPECTED_OCCURRENCES = 10_630
EXPECTED_ROWS = EXPECTED_SITES * EXPECTED_OCCURRENCES
EXPECTED_PARTITIONS = 16
EXPECTED_CATALOG_SHA256 = (
    "e4725a57eab466348aa263d79b8f8bcc923f9542fb5507af279fece41c121f37"
)
EXPECTED_SITE_ORDER_CRLF_SHA256 = (
    "fc64d53410fdadab5379f82c73c1d74ce3393911fbdd73a0dbc483f056f3d100"
)
EXPECTED_BASELINE_FIELDS_SHA256 = (
    "5d202929c997c262abfd068e47ba7c7fdba91873af8135d28dfec35b78193e5b"
)
EXPECTED_CROSS_IMT_RHO = 0.7321409900247263
EXPECTED_BASE_SEED = 20260731
EVENT_BATCH_SIZE = int(os.environ.get("NOTEBOOK9_EVENT_BATCH_SIZE", "64"))
GZIP_COMPRESSLEVEL = int(os.environ.get("NOTEBOOK9_GZIP_LEVEL", "1"))
VERIFY_PHASE1_CHUNK_HASHES = (
    os.environ.get("NOTEBOOK9_VERIFY_PHASE1_CHUNK_HASHES", "1") == "1"
)
if EVENT_BATCH_SIZE <= 0:
    raise ValueError("NOTEBOOK9_EVENT_BATCH_SIZE must be positive.")
if not 1 <= GZIP_COMPRESSLEVEL <= 9:
    raise ValueError("NOTEBOOK9_GZIP_LEVEL must lie between 1 and 9.")


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "09_generate_correlated_ground_motion_fields.ipynb").exists():
            return candidate
        if (
            (candidate / "tools" / "correlated_ground_motion.py").exists()
            and (candidate / "04_generate_ground_motion_fields.ipynb").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "Could not identify the seismic-correlation-insurance-loss repository root."
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from tools.correlated_ground_motion import (
    BASE_RANDOM_SEED,
    BASELINE_REQUIRED_COLUMNS,
    CASE_PREFIXES,
    COMMON_OUTPUT_COLUMNS,
    PAIRED_OUTPUT_COLUMNS,
    build_paired_field_batch,
    load_case_factors,
)
from tools.spatial_correlation import CASE_C1, CASE_C2, CASE_I0


def sha256_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(chunk_bytes):
            digest.update(block)
    return digest.hexdigest()


def canonical_crlf_sha256(path: Path) -> str:
    text = path.read_text(encoding="utf-8")
    normalized = "\r\n".join(text.splitlines()) + "\r\n"
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, indent=2, sort_keys=True, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)


def write_csv(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False, lineterminator="\n")
    temporary.replace(path)


def project_relative_path(path: Path) -> str:
    try:
        return path.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()
    except ValueError as exc:
        raise ValueError(f"Path is outside the project root: {path}") from exc


def resolve_repository_path(value: object) -> Path:
    text = str(value).strip().replace("\\", "/")
    marker = "/data/"
    if marker in text.lower():
        start = text.lower().index(marker) + 1
        return PROJECT_ROOT.joinpath(*text[start:].split("/"))
    path = Path(text).expanduser()
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    return path.resolve()


def normalize_boolean_series(values: pd.Series) -> pd.Series:
    return (
        values.astype("string")
        .fillna("")
        .str.strip()
        .str.lower()
        .isin({"true", "1", "yes", "y"})
    )


def add_check(
    rows: list[dict[str, object]],
    check_id: str,
    passed: bool,
    detail: str,
    severity: str = "critical",
) -> None:
    rows.append(
        {
            "check_id": check_id,
            "severity": severity,
            "passed": bool(passed),
            "detail": detail,
        }
    )


def combine_gzip_csv_files(
    source_paths: list[Path], destination_path: Path
) -> None:
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination_path.with_suffix(destination_path.suffix + ".tmp")
    expected_header: bytes | None = None
    with temporary.open("wb") as raw_destination:
        with gzip.GzipFile(
            fileobj=raw_destination,
            mode="wb",
            compresslevel=GZIP_COMPRESSLEVEL,
            mtime=0,
        ) as destination:
            for source_path in source_paths:
                with gzip.open(source_path, "rb") as source:
                    header = source.readline()
                    if expected_header is None:
                        expected_header = header
                        destination.write(header)
                    elif header != expected_header:
                        raise RuntimeError(
                            f"Inconsistent chunk header: {source_path}"
                        )
                    shutil.copyfileobj(source, destination, length=8 * 1024 * 1024)
    temporary.replace(destination_path)


class OnlineMoments:
    def __init__(self) -> None:
        self.count = 0
        self.total = 0.0
        self.total_sq = 0.0

    def update(self, values: Iterable[float]) -> None:
        array = np.asarray(values, dtype=np.float64)
        array = array[np.isfinite(array)]
        self.count += int(array.size)
        self.total += float(np.sum(array))
        self.total_sq += float(np.sum(array * array))

    @property
    def mean(self) -> float:
        return self.total / self.count if self.count else float("nan")

    @property
    def std(self) -> float:
        if self.count < 2:
            return float("nan")
        variance = (
            self.total_sq - self.count * self.mean**2
        ) / (self.count - 1)
        return math.sqrt(max(variance, 0.0))


class OnlineBivariate:
    def __init__(self) -> None:
        self.count = 0
        self.sx = 0.0
        self.sy = 0.0
        self.sxx = 0.0
        self.syy = 0.0
        self.sxy = 0.0

    def update(self, x: Iterable[float], y: Iterable[float]) -> None:
        first = np.asarray(x, dtype=np.float64)
        second = np.asarray(y, dtype=np.float64)
        mask = np.isfinite(first) & np.isfinite(second)
        first = first[mask]
        second = second[mask]
        self.count += int(first.size)
        self.sx += float(np.sum(first))
        self.sy += float(np.sum(second))
        self.sxx += float(np.sum(first * first))
        self.syy += float(np.sum(second * second))
        self.sxy += float(np.sum(first * second))

    @property
    def correlation(self) -> float:
        if self.count < 3:
            return float("nan")
        covariance = self.sxy - self.sx * self.sy / self.count
        variance_x = self.sxx - self.sx**2 / self.count
        variance_y = self.syy - self.sy**2 / self.count
        denominator = math.sqrt(max(variance_x * variance_y, 0.0))
        return covariance / denominator if denominator else float("nan")


DATA_DIR = PROJECT_ROOT / "data"
METADATA_DIR = DATA_DIR / "metadata"
PHASE1_HANDOFF_PATH = (
    METADATA_DIR
    / "notebook_4_final_handoff"
    / "notebook_5_input_handoff.json"
)
RANDOM_SPEC_PATH = (
    METADATA_DIR / "notebook_4_cell_19_random_stream_specification.json"
)
SITE_ORDER_PATH = METADATA_DIR / "notebook_4_cell_18_site_order.csv"
PHASE1_MANIFEST_PATH = METADATA_DIR / "notebook_4_cell_19_chunk_manifest.csv"
PHASE1_FINAL_FIELDS_PATH = (
    DATA_DIR
    / "processed"
    / "notebook_4_full_baseline_fields"
    / "full_baseline_ground_motion_fields.csv.gz"
)
NOTEBOOK8_HANDOFF_PATH = (
    METADATA_DIR
    / "phase_2"
    / "notebook_8_spatial_correlation"
    / "notebook_8_final_handoff.json"
)
FACTOR_PATH = (
    DATA_DIR
    / "processed"
    / "phase_2"
    / "notebook_8_spatial_correlation"
    / "spatial_correlation_factors.npz"
)
OUTPUT_DIR = (
    DATA_DIR
    / "processed"
    / "phase_2"
    / "notebook_9_correlated_ground_motion"
)
OUTPUT_FIELDS_PATH = OUTPUT_DIR / "paired_ground_motion_fields.csv.gz"
NOTEBOOK9_METADATA_DIR = (
    METADATA_DIR
    / "phase_2"
    / "notebook_9_correlated_ground_motion"
)
WORK_DIR = (
    METADATA_DIR
    / "phase_2"
    / "notebook_9_correlated_ground_motion_work"
)
CHUNK_DIR = WORK_DIR / "field_chunks"
VALIDATION_DIR = WORK_DIR / "chunk_validation"
MARKER_DIR = WORK_DIR / "completion_markers"
for directory in [OUTPUT_DIR, NOTEBOOK9_METADATA_DIR, CHUNK_DIR, VALIDATION_DIR, MARKER_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


required_input_paths = [
    PHASE1_HANDOFF_PATH,
    RANDOM_SPEC_PATH,
    SITE_ORDER_PATH,
    PHASE1_MANIFEST_PATH,
    PHASE1_FINAL_FIELDS_PATH,
    NOTEBOOK8_HANDOFF_PATH,
    FACTOR_PATH,
]
missing_input_paths = [
    project_relative_path(path)
    for path in required_input_paths
    if not path.is_file()
]
if missing_input_paths:
    raise FileNotFoundError(
        "Notebook 09 requires these validated upstream artifacts: "
        + ", ".join(missing_input_paths)
    )


phase1_handoff = load_json(PHASE1_HANDOFF_PATH)
random_spec = load_json(RANDOM_SPEC_PATH)
notebook8_handoff = load_json(NOTEBOOK8_HANDOFF_PATH)
site_order = pd.read_csv(SITE_ORDER_PATH)
phase1_manifest = pd.read_csv(PHASE1_MANIFEST_PATH, low_memory=False)
case_factors = load_case_factors(FACTOR_PATH)
site_ids = site_order.sort_values("site_ordinal")["site_id"].astype(str).tolist()

input_checks: list[dict[str, object]] = []
add_check(input_checks, "phase1_handoff_complete", phase1_handoff["notebook4_complete"] is True, "Notebook 04 handoff is complete.")
add_check(input_checks, "phase1_catalog_hash_frozen", phase1_handoff["annual_catalog"]["sha256"] == EXPECTED_CATALOG_SHA256, phase1_handoff["annual_catalog"]["sha256"])
add_check(input_checks, "phase1_occurrence_count_frozen", phase1_handoff["annual_catalog"]["occurrences"] == EXPECTED_OCCURRENCES, f"Occurrences: {phase1_handoff['annual_catalog']['occurrences']:,}.")
add_check(input_checks, "phase1_field_rows_frozen", phase1_handoff["notebook5_primary_input"]["rows"] == EXPECTED_ROWS, f"Rows: {phase1_handoff['notebook5_primary_input']['rows']:,}.")
add_check(input_checks, "phase1_field_hash_contract", phase1_handoff["notebook5_primary_input"]["sha256"] == EXPECTED_BASELINE_FIELDS_SHA256, phase1_handoff["notebook5_primary_input"]["sha256"])
add_check(input_checks, "phase1_baseline_file_exists", PHASE1_FINAL_FIELDS_PATH.is_file(), project_relative_path(PHASE1_FINAL_FIELDS_PATH))
add_check(input_checks, "phase1_random_seed_frozen", random_spec["base_random_seed"] == EXPECTED_BASE_SEED == BASE_RANDOM_SEED, f"Seed: {random_spec['base_random_seed']}.")
add_check(input_checks, "phase1_generator_frozen", random_spec["generator"] == "numpy.random.PCG64DXSM", random_spec["generator"])
add_check(input_checks, "phase1_site_seed_contract", random_spec["site_seed_parts"] == "base_random_seed | cell19 | occurrence_id | rupture_id | site", random_spec["site_seed_parts"])
add_check(input_checks, "cross_imt_rho_frozen", float(phase1_handoff["aleatory_model"]["cross_imt_rho"]) == EXPECTED_CROSS_IMT_RHO, f"rho: {phase1_handoff['aleatory_model']['cross_imt_rho']:.16f}.")
add_check(input_checks, "site_order_rows", len(site_order) == EXPECTED_SITES, f"Sites: {len(site_order):,}.")
add_check(input_checks, "site_order_hash_frozen", canonical_crlf_sha256(SITE_ORDER_PATH) == EXPECTED_SITE_ORDER_CRLF_SHA256, canonical_crlf_sha256(SITE_ORDER_PATH))
add_check(input_checks, "notebook8_complete", notebook8_handoff["notebook8_complete"] is True, notebook8_handoff["schema_version"])
add_check(input_checks, "notebook8_cases_complete", set(notebook8_handoff["cases"]) == set(CASE_PREFIXES), str(notebook8_handoff["cases"]))
add_check(input_checks, "notebook8_factor_exists", FACTOR_PATH.is_file(), project_relative_path(FACTOR_PATH))
add_check(input_checks, "notebook8_factor_hash", sha256_file(FACTOR_PATH) == notebook8_handoff["factor_artifact"]["sha256"], sha256_file(FACTOR_PATH))
add_check(input_checks, "phase1_partition_count", len(phase1_manifest) == EXPECTED_PARTITIONS, f"Partitions: {len(phase1_manifest)}.")
add_check(input_checks, "phase1_partition_occurrences", int(phase1_manifest["catalog_occurrence_count"].sum()) == EXPECTED_OCCURRENCES, f"Occurrences: {int(phase1_manifest['catalog_occurrence_count'].sum()):,}.")
add_check(input_checks, "phase1_partition_rows", int(phase1_manifest["expected_field_rows"].sum()) == EXPECTED_ROWS, f"Rows: {int(phase1_manifest['expected_field_rows'].sum()):,}.")

if PHASE1_FINAL_FIELDS_PATH.is_file():
    phase1_final_hash = sha256_file(PHASE1_FINAL_FIELDS_PATH)
    add_check(input_checks, "phase1_final_file_hash", phase1_final_hash == EXPECTED_BASELINE_FIELDS_SHA256, phase1_final_hash)

input_validation = pd.DataFrame(input_checks)
input_validation_path = NOTEBOOK9_METADATA_DIR / "notebook_9_input_validation.csv"
write_csv(input_validation, input_validation_path)
if not normalize_boolean_series(input_validation["passed"]).all():
    display(input_validation.loc[~normalize_boolean_series(input_validation["passed"])])
    raise RuntimeError("Notebook 09 frozen-input validation failed.")

print("=" * 78)
print("NOTEBOOK 09 CELL 1: FROZEN INPUTS VALIDATED")
print("=" * 78)
print(f"Catalog occurrences:          {EXPECTED_OCCURRENCES:,}")
print(f"Event-site rows:              {EXPECTED_ROWS:,}")
print(f"Phase 1 partitions:           {len(phase1_manifest):,}")
print(f"Event batch size:             {EVENT_BATCH_SIZE:,}")
print(f"Notebook 08 factor:           {project_relative_path(FACTOR_PATH)}")
print(f"Critical checks:              {len(input_validation):,}")
print("Next: resolve and verify the 16 restartable Phase 1 field partitions.")

## Paired transformation

For every occurrence, the two Phase 1 site vectors are regenerated from
the frozen `site_seed`. The I0 result must reproduce Phase 1 exactly.
C1 and C2 apply their validated spatial roots before rebuilding ground
motion through

$$
\ln(IM_{e,s})=\mu_{e,s}+\tau_{e,s}\eta_e+\phi_{e,s}\epsilon_{e,s}.
$$

A single wide row stores all three cases. This retains direct pairing
while avoiding three copies of the event and site identifiers.

In [ ]:
required_manifest_columns = {
    "chunk_number",
    "chunk_id",
    "catalog_occurrence_count",
    "expected_field_rows",
    "field_output",
    "marker_path",
}
missing_manifest_columns = sorted(
    required_manifest_columns - set(phase1_manifest.columns)
)
if missing_manifest_columns:
    raise RuntimeError(
        f"Phase 1 manifest is missing columns: {missing_manifest_columns}"
    )

partition_records: list[dict[str, object]] = []
partition_checks: list[dict[str, object]] = []
for row in phase1_manifest.sort_values("chunk_number").itertuples(index=False):
    chunk_number = int(row.chunk_number)
    chunk_id = f"{chunk_number:04d}"
    input_path = resolve_repository_path(row.field_output)
    phase1_marker_path = resolve_repository_path(row.marker_path)
    output_path = CHUNK_DIR / f"paired_fields_chunk_{chunk_id}.csv.gz"
    validation_path = VALIDATION_DIR / f"paired_fields_chunk_{chunk_id}_validation.csv"
    marker_path = MARKER_DIR / f"paired_fields_chunk_{chunk_id}.complete.json"
    expected_occurrences = int(row.catalog_occurrence_count)
    expected_rows = int(row.expected_field_rows)

    input_hash = ""
    marker_hash = ""
    marker_valid = False
    if input_path.is_file() and phase1_marker_path.is_file():
        phase1_marker = load_json(phase1_marker_path)
        marker_hash = str(phase1_marker.get("field_output_sha256", ""))
        if VERIFY_PHASE1_CHUNK_HASHES:
            input_hash = sha256_file(input_path)
            marker_valid = input_hash == marker_hash
        else:
            input_hash = marker_hash
            marker_valid = bool(marker_hash)

    add_check(partition_checks, f"partition_{chunk_id}_input_exists", input_path.is_file(), project_relative_path(input_path))
    add_check(partition_checks, f"partition_{chunk_id}_marker_exists", phase1_marker_path.is_file(), project_relative_path(phase1_marker_path))
    add_check(partition_checks, f"partition_{chunk_id}_hash", marker_valid, f"Observed: {input_hash}; expected: {marker_hash}.")
    add_check(partition_checks, f"partition_{chunk_id}_row_contract", expected_rows == expected_occurrences * EXPECTED_SITES, f"Rows: {expected_rows:,}; occurrences: {expected_occurrences:,}.")

    partition_records.append(
        {
            "chunk_number": chunk_number,
            "chunk_id": chunk_id,
            "expected_occurrences": expected_occurrences,
            "expected_rows": expected_rows,
            "phase1_input_path": input_path,
            "phase1_input_sha256": input_hash,
            "output_path": output_path,
            "validation_path": validation_path,
            "marker_path": marker_path,
        }
    )

partition_validation = pd.DataFrame(partition_checks)
partition_validation_path = NOTEBOOK9_METADATA_DIR / "notebook_9_partition_validation.csv"
write_csv(partition_validation, partition_validation_path)
if not normalize_boolean_series(partition_validation["passed"]).all():
    display(partition_validation.loc[~normalize_boolean_series(partition_validation["passed"])])
    raise RuntimeError(
        "Notebook 09 cannot start because a Phase 1 restart partition is missing or invalid."
    )

print("=" * 78)
print("NOTEBOOK 09 CELL 2: PHASE 1 PARTITIONS VERIFIED")
print("=" * 78)
print(f"Partitions:                   {len(partition_records):,}")
print(f"Occurrences:                  {sum(int(r['expected_occurrences']) for r in partition_records):,}")
print(f"Rows:                         {sum(int(r['expected_rows']) for r in partition_records):,}")
print(f"Chunk hashes verified:        {VERIFY_PHASE1_CHUNK_HASHES}")
print("Next: run a one-occurrence controlled paired reconstruction.")

In [ ]:
control_record = partition_records[0]
control_input = pd.read_csv(
    control_record["phase1_input_path"],
    nrows=EXPECTED_SITES,
    usecols=BASELINE_REQUIRED_COLUMNS,
    dtype={
        "catalog_event_id": str,
        "occurrence_id": str,
        "rupture_template_event_id": str,
        "rupture_id": str,
        "site_id": str,
        "source_type": str,
        "gmm_name": str,
    },
    low_memory=False,
)
control_output, control_diagnostics = build_paired_field_batch(
    control_input,
    case_factors,
    expected_site_ids=site_ids,
)
control_checks: list[dict[str, object]] = []
add_check(control_checks, "control_rows", len(control_output) == EXPECTED_SITES, f"Rows: {len(control_output)}.")
add_check(control_checks, "control_i0_epsilon_pga", control_diagnostics.maximum_i0_epsilon_pga_error <= 3.0e-12, f"Error: {control_diagnostics.maximum_i0_epsilon_pga_error:.6e}.")
add_check(control_checks, "control_i0_epsilon_sa0p4", control_diagnostics.maximum_i0_epsilon_sa0p4_error <= 3.0e-12, f"Error: {control_diagnostics.maximum_i0_epsilon_sa0p4_error:.6e}.")
add_check(control_checks, "control_i0_pga_ln", control_diagnostics.maximum_i0_pga_ln_error <= 3.0e-12, f"Error: {control_diagnostics.maximum_i0_pga_ln_error:.6e}.")
add_check(control_checks, "control_i0_sa0p4_ln", control_diagnostics.maximum_i0_sa0p4_ln_error <= 3.0e-12, f"Error: {control_diagnostics.maximum_i0_sa0p4_ln_error:.6e}.")
add_check(control_checks, "control_outputs_finite", control_diagnostics.nonfinite_output_count == 0, f"Nonfinite values: {control_diagnostics.nonfinite_output_count}.")
add_check(control_checks, "control_c1_differs_from_i0", bool(np.any(control_output["c1_pga_simulated_ln_g"].to_numpy() != control_output["i0_pga_simulated_ln_g"].to_numpy())), "C1 changes within-event dependence.")
add_check(control_checks, "control_c2_differs_from_i0", bool(np.any(control_output["c2_pga_simulated_ln_g"].to_numpy() != control_output["i0_pga_simulated_ln_g"].to_numpy())), "C2 changes within-event dependence.")

control_validation = pd.DataFrame(control_checks)
control_validation_path = NOTEBOOK9_METADATA_DIR / "notebook_9_control_validation.csv"
write_csv(control_validation, control_validation_path)
if not normalize_boolean_series(control_validation["passed"]).all():
    display(control_validation.loc[~normalize_boolean_series(control_validation["passed"])])
    raise RuntimeError("Notebook 09 controlled reconstruction failed.")

print("=" * 78)
print("NOTEBOOK 09 CELL 3: CONTROLLED RECONSTRUCTION PASSED")
print("=" * 78)
print(f"Occurrence:                   {control_output['occurrence_id'].iloc[0]}")
print(f"Sites:                        {len(control_output):,}")
print(f"I0 PGA epsilon error:         {control_diagnostics.maximum_i0_epsilon_pga_error:.6e}")
print(f"I0 SA0P4 epsilon error:       {control_diagnostics.maximum_i0_epsilon_sa0p4_error:.6e}")
print("Next: generate or resume all 16 paired field partitions.")

## Restartable full-catalog generation

Each Phase 1 partition is processed in complete occurrence batches.
Completion markers contain the frozen input hash, factor hash, output
hash, row count, and validation status. A valid partition is reused on
restart. An invalid or incomplete partition is regenerated atomically.

In [ ]:
factor_sha256 = sha256_file(FACTOR_PATH)
module_sha256 = sha256_file(PROJECT_ROOT / "tools" / "correlated_ground_motion.py")


def marker_is_valid(record: dict[str, object]) -> bool:
    marker_path = Path(record["marker_path"])
    output_path = Path(record["output_path"])
    validation_path = Path(record["validation_path"])
    if not all(path.is_file() for path in [marker_path, output_path, validation_path]):
        return False
    try:
        marker = load_json(marker_path)
        validation = pd.read_csv(validation_path)
    except (OSError, json.JSONDecodeError, pd.errors.ParserError):
        return False
    if "passed" not in validation.columns:
        return False
    required = {
        "pipeline_version": PIPELINE_VERSION,
        "phase1_input_sha256": record["phase1_input_sha256"],
        "factor_sha256": factor_sha256,
        "module_sha256": module_sha256,
        "expected_rows": int(record["expected_rows"]),
        "expected_occurrences": int(record["expected_occurrences"]),
    }
    if any(marker.get(key) != value for key, value in required.items()):
        return False
    if not normalize_boolean_series(validation["passed"]).all():
        return False
    if marker.get("validation_sha256") != sha256_file(validation_path):
        return False
    return marker.get("output_sha256") == sha256_file(output_path)


def generate_partition(record: dict[str, object]) -> pd.DataFrame:
    if marker_is_valid(record):
        return pd.read_csv(record["validation_path"])

    output_path = Path(record["output_path"])
    validation_path = Path(record["validation_path"])
    marker_path = Path(record["marker_path"])
    for path in [output_path, validation_path, marker_path]:
        path.unlink(missing_ok=True)
    temporary_output = output_path.with_suffix(output_path.suffix + ".tmp")

    maxima = defaultdict(float)
    observed_rows = 0
    observed_occurrences = 0
    header_written = False
    read_rows = EVENT_BATCH_SIZE * EXPECTED_SITES
    reader = pd.read_csv(
        record["phase1_input_path"],
        usecols=BASELINE_REQUIRED_COLUMNS,
        dtype={
            "catalog_event_id": str,
            "occurrence_id": str,
            "rupture_template_event_id": str,
            "rupture_id": str,
            "site_id": str,
            "source_type": str,
            "gmm_name": str,
        },
        chunksize=read_rows,
        low_memory=False,
    )
    with temporary_output.open("wb") as raw:
        with gzip.GzipFile(
            fileobj=raw,
            mode="wb",
            compresslevel=GZIP_COMPRESSLEVEL,
            mtime=0,
        ) as compressed:
            for baseline_batch in reader:
                paired_batch, diagnostics = build_paired_field_batch(
                    baseline_batch,
                    case_factors,
                    expected_site_ids=site_ids,
                )
                encoded = paired_batch.to_csv(
                    index=False,
                    header=not header_written,
                    na_rep="",
                    lineterminator="\n",
                    float_format="%.17g",
                ).encode("utf-8")
                compressed.write(encoded)
                header_written = True
                observed_rows += diagnostics.rows
                observed_occurrences += diagnostics.occurrences
                for key, value in diagnostics.to_dict().items():
                    if key not in {"rows", "occurrences", "site_count"}:
                        maxima[key] = max(maxima[key], float(value))
    temporary_output.replace(output_path)

    checks: list[dict[str, object]] = []
    add_check(checks, "row_count", observed_rows == int(record["expected_rows"]), f"Observed: {observed_rows:,}; expected: {int(record['expected_rows']):,}.")
    add_check(checks, "occurrence_count", observed_occurrences == int(record["expected_occurrences"]), f"Observed: {observed_occurrences:,}; expected: {int(record['expected_occurrences']):,}.")
    add_check(checks, "i0_epsilon_pga", maxima["maximum_i0_epsilon_pga_error"] <= 3.0e-12, f"Error: {maxima['maximum_i0_epsilon_pga_error']:.6e}.")
    add_check(checks, "i0_epsilon_sa0p4", maxima["maximum_i0_epsilon_sa0p4_error"] <= 3.0e-12, f"Error: {maxima['maximum_i0_epsilon_sa0p4_error']:.6e}.")
    add_check(checks, "i0_pga_ln", maxima["maximum_i0_pga_ln_error"] <= 3.0e-12, f"Error: {maxima['maximum_i0_pga_ln_error']:.6e}.")
    add_check(checks, "i0_sa0p4_ln", maxima["maximum_i0_sa0p4_ln_error"] <= 3.0e-12, f"Error: {maxima['maximum_i0_sa0p4_ln_error']:.6e}.")
    add_check(checks, "eta_pga_repeated", maxima["maximum_repeated_eta_pga_error"] == 0.0, f"Error: {maxima['maximum_repeated_eta_pga_error']:.6e}.")
    add_check(checks, "eta_sa0p4_repeated", maxima["maximum_repeated_eta_sa0p4_error"] == 0.0, f"Error: {maxima['maximum_repeated_eta_sa0p4_error']:.6e}.")
    add_check(checks, "outputs_finite", maxima["nonfinite_output_count"] == 0.0, f"Nonfinite values: {int(maxima['nonfinite_output_count'])}.")
    validation = pd.DataFrame(checks)
    write_csv(validation, validation_path)
    if not normalize_boolean_series(validation["passed"]).all():
        output_path.unlink(missing_ok=True)
        display(validation.loc[~normalize_boolean_series(validation["passed"])])
        raise RuntimeError(f"Partition {record['chunk_id']} failed validation.")

    output_sha256 = sha256_file(output_path)
    write_json(
        marker_path,
        {
            "pipeline_version": PIPELINE_VERSION,
            "chunk_id": record["chunk_id"],
            "phase1_input_sha256": record["phase1_input_sha256"],
            "factor_sha256": factor_sha256,
            "module_sha256": module_sha256,
            "expected_rows": int(record["expected_rows"]),
            "expected_occurrences": int(record["expected_occurrences"]),
            "output_sha256": output_sha256,
            "output_bytes": int(output_path.stat().st_size),
            "validation_sha256": sha256_file(validation_path),
        },
    )
    return validation


partition_validation_tables: list[pd.DataFrame] = []
manifest_rows: list[dict[str, object]] = []
for position, record in enumerate(partition_records, start=1):
    print(
        f"Partition {position:02d}/{len(partition_records):02d} "
        f"({record['chunk_id']}): {int(record['expected_occurrences']):,} occurrences"
    )
    validation = generate_partition(record)
    validation.insert(0, "chunk_id", record["chunk_id"])
    partition_validation_tables.append(validation)
    marker = load_json(Path(record["marker_path"]))
    manifest_rows.append(
        {
            "chunk_number": int(record["chunk_number"]),
            "chunk_id": record["chunk_id"],
            "occurrences": int(record["expected_occurrences"]),
            "rows": int(record["expected_rows"]),
            "phase1_input_path": project_relative_path(Path(record["phase1_input_path"])),
            "phase1_input_sha256": record["phase1_input_sha256"],
            "output_path": project_relative_path(Path(record["output_path"])),
            "output_sha256": marker["output_sha256"],
            "output_bytes": marker["output_bytes"],
            "validation_path": project_relative_path(Path(record["validation_path"])),
            "validation_sha256": marker["validation_sha256"],
            "status": "complete",
        }
    )
    write_csv(
        pd.DataFrame(manifest_rows),
        NOTEBOOK9_METADATA_DIR / "notebook_9_chunk_manifest.csv",
    )

all_partition_validation = pd.concat(
    partition_validation_tables, ignore_index=True
)
write_csv(
    all_partition_validation,
    NOTEBOOK9_METADATA_DIR / "notebook_9_chunk_validation.csv",
)
if not normalize_boolean_series(all_partition_validation["passed"]).all():
    raise RuntimeError("At least one Notebook 09 partition failed validation.")

print("=" * 78)
print("NOTEBOOK 09 CELL 4: ALL PAIRED PARTITIONS COMPLETE")
print("=" * 78)
print(f"Partitions:                   {len(manifest_rows):,}")
print(f"Occurrences:                  {sum(row['occurrences'] for row in manifest_rows):,}")
print(f"Rows:                         {sum(row['rows'] for row in manifest_rows):,}")
print("Next: combine partitions and validate full-catalog dependence.")

In [ ]:
ordered_chunk_paths = [
    Path(record["output_path"])
    for record in sorted(partition_records, key=lambda item: int(item["chunk_number"]))
]
combine_gzip_csv_files(ordered_chunk_paths, OUTPUT_FIELDS_PATH)
output_sha256 = sha256_file(OUTPUT_FIELDS_PATH)

with np.load(FACTOR_PATH, allow_pickle=False) as factor_payload:
    distance_km = np.asarray(factor_payload["distance_km"], dtype=np.float64)
    target_correlations = {
        "i0": {
            "pga": np.asarray(factor_payload["i0_pga_correlation"], dtype=np.float64),
            "sa0p4": np.asarray(factor_payload["i0_sa0p4_correlation"], dtype=np.float64),
        },
        "c1": {
            "pga": np.asarray(factor_payload["c1_pga_correlation"], dtype=np.float64),
            "sa0p4": np.asarray(factor_payload["c1_sa0p4_correlation"], dtype=np.float64),
        },
        "c2": {
            "pga": np.asarray(factor_payload["c2_pga_correlation"], dtype=np.float64),
            "sa0p4": np.asarray(factor_payload["c2_sa0p4_correlation"], dtype=np.float64),
        },
    }

upper_i, upper_j = np.triu_indices(EXPECTED_SITES, k=1)
upper_distances = distance_km[upper_i, upper_j]
selected_pairs: list[tuple[int, int]] = []
for probability in [0.00, 0.05, 0.25, 0.50, 0.75, 0.95, 1.00]:
    target_distance = float(np.quantile(upper_distances, probability))
    candidate = int(np.argmin(np.abs(upper_distances - target_distance)))
    pair = (int(upper_i[candidate]), int(upper_j[candidate]))
    if pair not in selected_pairs:
        selected_pairs.append(pair)

epsilon_columns = [
    f"{prefix}_epsilon_{imt}"
    for prefix in CASE_PREFIXES.values()
    for imt in ["pga", "sa0p4"]
]
moments = {column: OnlineMoments() for column in epsilon_columns}
cross_imt_stats = {
    prefix: OnlineBivariate() for prefix in CASE_PREFIXES.values()
}
pair_stats = {
    (prefix, imt, site_i, site_j): OnlineBivariate()
    for prefix in CASE_PREFIXES.values()
    for imt in ["pga", "sa0p4"]
    for site_i, site_j in selected_pairs
}
final_rows = 0
final_occurrences = 0
read_columns = ["occurrence_id", "site_ordinal", *epsilon_columns]
for field_batch in pd.read_csv(
    OUTPUT_FIELDS_PATH,
    usecols=read_columns,
    dtype={"occurrence_id": str},
    chunksize=EVENT_BATCH_SIZE * EXPECTED_SITES,
    low_memory=False,
):
    if len(field_batch) % EXPECTED_SITES != 0:
        raise RuntimeError("Final output scan encountered an incomplete occurrence.")
    occurrences = len(field_batch) // EXPECTED_SITES
    expected_ordinals = np.tile(np.arange(EXPECTED_SITES), occurrences)
    if not np.array_equal(field_batch["site_ordinal"].to_numpy(dtype=int), expected_ordinals):
        raise RuntimeError("Final output scan found invalid site ordering.")
    final_rows += len(field_batch)
    final_occurrences += occurrences
    arrays = {
        column: field_batch[column].to_numpy(dtype=np.float64).reshape(occurrences, EXPECTED_SITES)
        for column in epsilon_columns
    }
    for column, array in arrays.items():
        moments[column].update(array.reshape(-1))
    for prefix in CASE_PREFIXES.values():
        cross_imt_stats[prefix].update(
            arrays[f"{prefix}_epsilon_pga"].reshape(-1),
            arrays[f"{prefix}_epsilon_sa0p4"].reshape(-1),
        )
        for imt in ["pga", "sa0p4"]:
            values = arrays[f"{prefix}_epsilon_{imt}"]
            for site_i, site_j in selected_pairs:
                pair_stats[(prefix, imt, site_i, site_j)].update(
                    values[:, site_i], values[:, site_j]
                )

case_summary_rows: list[dict[str, object]] = []
for case_name, prefix in CASE_PREFIXES.items():
    for imt in ["pga", "sa0p4"]:
        statistic = moments[f"{prefix}_epsilon_{imt}"]
        case_summary_rows.append(
            {
                "case_name": case_name,
                "case_prefix": prefix,
                "imt": imt.upper(),
                "residual_count": statistic.count,
                "residual_mean": statistic.mean,
                "residual_standard_deviation": statistic.std,
                "same_site_cross_imt_correlation": cross_imt_stats[prefix].correlation,
            }
        )
case_summary = pd.DataFrame(case_summary_rows)
case_summary_path = NOTEBOOK9_METADATA_DIR / "notebook_9_case_summary.csv"
write_csv(case_summary, case_summary_path)

pair_rows: list[dict[str, object]] = []
for case_name, prefix in CASE_PREFIXES.items():
    for imt in ["pga", "sa0p4"]:
        target_matrix = target_correlations[prefix][imt]
        for site_i, site_j in selected_pairs:
            empirical = pair_stats[(prefix, imt, site_i, site_j)].correlation
            target = float(target_matrix[site_i, site_j])
            pair_rows.append(
                {
                    "case_name": case_name,
                    "case_prefix": prefix,
                    "imt": imt.upper(),
                    "site_i": site_i,
                    "site_j": site_j,
                    "distance_km": float(distance_km[site_i, site_j]),
                    "target_correlation": target,
                    "empirical_correlation": empirical,
                    "absolute_error": abs(empirical - target),
                }
            )
pair_diagnostics = pd.DataFrame(pair_rows)
pair_diagnostics_path = NOTEBOOK9_METADATA_DIR / "notebook_9_pair_diagnostics.csv"
write_csv(pair_diagnostics, pair_diagnostics_path)

final_checks: list[dict[str, object]] = []
add_check(final_checks, "partition_validations_passed", normalize_boolean_series(all_partition_validation["passed"]).all(), f"Checks: {len(all_partition_validation):,}.")
add_check(final_checks, "final_row_count", final_rows == EXPECTED_ROWS, f"Observed: {final_rows:,}; expected: {EXPECTED_ROWS:,}.")
add_check(final_checks, "final_occurrence_count", final_occurrences == EXPECTED_OCCURRENCES, f"Observed: {final_occurrences:,}; expected: {EXPECTED_OCCURRENCES:,}.")
add_check(final_checks, "output_schema", pd.read_csv(OUTPUT_FIELDS_PATH, nrows=0).columns.tolist() == PAIRED_OUTPUT_COLUMNS, f"Columns: {len(PAIRED_OUTPUT_COLUMNS)}.")
add_check(final_checks, "residual_means", bool(case_summary["residual_mean"].abs().le(0.05).all()), f"Maximum absolute mean: {case_summary['residual_mean'].abs().max():.6f}.")
add_check(final_checks, "residual_standard_deviations", bool(case_summary["residual_standard_deviation"].between(0.97, 1.03).all()), f"Range: {case_summary['residual_standard_deviation'].min():.6f} to {case_summary['residual_standard_deviation'].max():.6f}.")
add_check(final_checks, "same_site_cross_imt_correlation", bool((case_summary["same_site_cross_imt_correlation"] - EXPECTED_CROSS_IMT_RHO).abs().le(0.02).all()), f"Maximum error: {(case_summary['same_site_cross_imt_correlation'] - EXPECTED_CROSS_IMT_RHO).abs().max():.6f}.")
add_check(final_checks, "selected_pair_correlations", bool(pair_diagnostics["absolute_error"].le(0.04).all()), f"Maximum error: {pair_diagnostics['absolute_error'].max():.6f}.")
add_check(final_checks, "metadata_paths_portable", all(not Path(path).is_absolute() for path in [project_relative_path(OUTPUT_FIELDS_PATH), project_relative_path(case_summary_path), project_relative_path(pair_diagnostics_path)]), "All public paths are repository-relative.")
final_validation = pd.DataFrame(final_checks)
final_validation_path = NOTEBOOK9_METADATA_DIR / "notebook_9_final_validation.csv"
write_csv(final_validation, final_validation_path)
if not normalize_boolean_series(final_validation["passed"]).all():
    display(final_validation.loc[~normalize_boolean_series(final_validation["passed"])])
    raise RuntimeError("Notebook 09 final validation failed.")

print("=" * 78)
print("NOTEBOOK 09 CELL 5: FULL-CATALOG DEPENDENCE VALIDATED")
print("=" * 78)
print(f"Output rows:                  {final_rows:,}")
print(f"Output occurrences:           {final_occurrences:,}")
print(f"Maximum pair error:           {pair_diagnostics['absolute_error'].max():.6f}")
print(f"Output SHA-256:               {output_sha256}")
print("Next: write the restartable Notebook 10 handoff.")
display(case_summary)

In [ ]:
chunk_manifest_path = NOTEBOOK9_METADATA_DIR / "notebook_9_chunk_manifest.csv"
chunk_validation_path = NOTEBOOK9_METADATA_DIR / "notebook_9_chunk_validation.csv"
artifact_paths = [
    input_validation_path,
    partition_validation_path,
    control_validation_path,
    chunk_manifest_path,
    chunk_validation_path,
    case_summary_path,
    pair_diagnostics_path,
    final_validation_path,
    OUTPUT_FIELDS_PATH,
]
artifact_inventory = [
    {
        "path": project_relative_path(path),
        "sha256": sha256_file(path),
        "bytes": int(path.stat().st_size),
    }
    for path in artifact_paths
]
final_handoff = {
    "schema_version": "notebook9_correlated_ground_motion_handoff_v1",
    "pipeline_version": PIPELINE_VERSION,
    "notebook9_complete": True,
    "frozen_controls": {
        "phase1_release": "v1.0.0",
        "phase1_commit": "be93474ce2ab78d8002d49ae861adb641ae2741d",
        "catalog_sha256": EXPECTED_CATALOG_SHA256,
        "baseline_fields_sha256": EXPECTED_BASELINE_FIELDS_SHA256,
        "site_order_canonical_crlf_sha256": EXPECTED_SITE_ORDER_CRLF_SHA256,
        "base_random_seed": EXPECTED_BASE_SEED,
        "random_generator": "numpy.random.PCG64DXSM",
        "cross_imt_rho": EXPECTED_CROSS_IMT_RHO,
        "catalog_years": 2_000_000,
        "occurrences": EXPECTED_OCCURRENCES,
        "sites": EXPECTED_SITES,
    },
    "dependence_cases": list(CASE_PREFIXES),
    "paired_output": {
        "path": project_relative_path(OUTPUT_FIELDS_PATH),
        "sha256": output_sha256,
        "rows": final_rows,
        "columns": PAIRED_OUTPUT_COLUMNS,
        "row_granularity": "one catalog occurrence and one site",
    },
    "factor_artifact": {
        "path": project_relative_path(FACTOR_PATH),
        "sha256": factor_sha256,
    },
    "validation": {
        "path": project_relative_path(final_validation_path),
        "sha256": sha256_file(final_validation_path),
        "checks": int(len(final_validation)),
        "critical_failures": int((~normalize_boolean_series(final_validation["passed"])).sum()),
        "maximum_selected_pair_correlation_error": float(pair_diagnostics["absolute_error"].max()),
    },
    "artifact_inventory": artifact_inventory,
    "next_notebook": "10_correlated_damage_and_loss.ipynb",
    "next_task": (
        "Apply the frozen structural and nonstructural damage uniforms to "
        "the paired I0, C1, and C2 ground-motion fields, then calculate "
        "ground-up and insured losses without changing exposure or policy terms."
    ),
}
final_handoff_path = NOTEBOOK9_METADATA_DIR / "notebook_9_final_handoff.json"
write_json(final_handoff_path, final_handoff)

print("=" * 78)
print("NOTEBOOK 09 COMPLETE: FULL-CATALOG PAIRED GROUND MOTIONS VALIDATED")
print("=" * 78)
print(f"Dependence cases:             {len(CASE_PREFIXES):,}")
print(f"Catalog occurrences:          {EXPECTED_OCCURRENCES:,}")
print(f"Event-site rows:              {final_rows:,}")
print(f"Validation checks:            {len(final_validation):,}")
print("Critical failures:            0")
print(f"Paired field:                 {project_relative_path(OUTPUT_FIELDS_PATH)}")
print(f"Final handoff:                {project_relative_path(final_handoff_path)}")
print("Next: Notebook 10 correlated structural and nonstructural damage and loss.")